In [1]:
!pip install torch torchvision opencv-python albumentations matplotlib scikit-learn pillow pydicom pynrrd

Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# Run this to understand your data better
import json
from pathlib import Path
import nrrd
import numpy as np

for patient_dir in list(Path('exported_patients/train').iterdir())[:3]:
    print(f"\n{'='*60}")
    print(f"Patient: {patient_dir.name}")
    print('='*60)
    
    # Check JSON
    with open(patient_dir / 'patient_data.json') as f:
        data = json.load(f)
    
    print(f"\nClinical Data:")
    print(f"  Tumor Grade: {data['demographic_clinical'].get('Tumor Grade')}")
    print(f"  Staging: T{data['demographic_clinical'].get('Staging(Tumor Size)#[T]')}")
    print(f"  ER/PR/HER2: {data['demographic_clinical'].get('ER')}/{data['demographic_clinical'].get('PR')}/{data['demographic_clinical'].get('HER2')}")
    
    print(f"\nRadiomic Features:")
    print(f"  Tumor Volume: {data['imaging_features'].get('Volume_cu_mm_Tumor')} mm³")
    print(f"  Tumor Size: {data['imaging_features'].get('TumorMajorAxisLength_mm')} mm")
    
    print(f"\nAnnotations:")
    annotations = data.get('annotations', [])
    print(f"  Number of annotations: {len(annotations)}")
    if annotations:
        print(f"  First annotation keys: {list(annotations[0].keys())}")
    
    print(f"\nMasks:")
    seg_dir = patient_dir / 'Segmentation_Masks_NRRD'
    if seg_dir.exists():
        for mask_file in seg_dir.glob('*.nrrd'):
            mask_data, _ = nrrd.read(str(mask_file))
            coverage = (mask_data > 0).sum() / mask_data.size * 100
            print(f"  {mask_file.name}: {mask_data.shape}, {coverage:.1f}% annotated")


Patient: Patient_099

Clinical Data:
  Tumor Grade: 3
  Staging: TNone
  ER/PR/HER2: 0/0/0

Radiomic Features:
  Tumor Volume: 3317.96875 mm³
  Tumor Size: 42.5787503686194 mm

Annotations:
  Number of annotations: 1
  First annotation keys: ['patient_id', 'bounding_box_3d', 'annotation_type']

Masks:

Patient: Patient_431

Clinical Data:
  Tumor Grade: 2
  Staging: TNone
  ER/PR/HER2: 0/0/1

Radiomic Features:
  Tumor Volume: 1402.82349324318 mm³
  Tumor Size: 47.6337706136036 mm

Annotations:
  Number of annotations: 1
  First annotation keys: ['patient_id', 'bounding_box_3d', 'annotation_type']

Masks:

Patient: Patient_375

Clinical Data:
  Tumor Grade: 1
  Staging: TNone
  ER/PR/HER2: 0/0/0

Radiomic Features:
  Tumor Volume: 882.390046645795 mm³
  Tumor Size: 18.6067770008213 mm

Annotations:
  Number of annotations: 1
  First annotation keys: ['patient_id', 'bounding_box_3d', 'annotation_type']

Masks:
  v2_breast_train_Breast_MRI_375_pre_050.nrrd: (448, 448, 1), 7.3% annotated

In [17]:
!python data_distribution_analysis.py


DUKE DATASET DISTRIBUTION ANALYSIS

ANALYZING: exported_patients/train
Total patients: 671

📊 CLASS DISTRIBUTION:

  Tumor Grade:
    Grade 1:  51 (  7.6%)
    Grade 2:  98 ( 14.6%)
    Grade 3: 522 ( 77.8%)

  Molecular Subtype:
    Triple Negative     : 118 ( 17.6%)
    HER2+               : 119 ( 17.7%)
    Luminal             : 434 ( 64.7%)

  Receptor Status:
    ER+:  502 ( 74.8%)
    PR+:  443 ( 66.0%)
    HER2+: 119 ( 17.7%)

  High Grade (Grade 3):
    Yes: 522 ( 77.8%)
    No:  149 ( 22.2%)

  3D Bounding Boxes: 671 (100.0%)

📈 CLINICAL FEATURES:
   Field Strength: 1.94 ± 1.00 T
   TR (ms):        5 ± 1
   TE (ms):        2 ± 0

🩺 IMAGING FEATURES:
   Tumor Size:   38.6 ± 31.7 mm
   Tumor Volume: 10745 ± 26741 mm³

ANALYZING: exported_patients/test
Total patients: 169

📊 CLASS DISTRIBUTION:

  Tumor Grade:
    Grade 1:  13 (  7.7%)
    Grade 2:  28 ( 16.6%)
    Grade 3: 128 ( 75.7%)

  Molecular Subtype:
    Triple Negative     :  29 ( 17.2%)
    HER2+               :  33 ( 

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

        # alpha can be:
        # - None
        # - scalar (float/int)
        # - list/tuple/tensor of class weights
        if alpha is None:
            self.alpha = None
        elif isinstance(alpha, (float, int)):
            self.alpha = torch.tensor(float(alpha), dtype=torch.float32)
        else:
            self.alpha = torch.tensor(alpha, dtype=torch.float32)

    def forward(self, inputs, targets):
        # inputs: [N, C], targets: [N]
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")  # [N]
        pt = torch.exp(-ce_loss)  # [N]
        focal_term = (1.0 - pt) ** self.gamma  # [N]

        if self.alpha is None:
            alpha_t = 1.0
        else:
            alpha = self.alpha.to(inputs.device)
            if alpha.ndim == 0:
                alpha_t = alpha
            else:
                # pick class-specific alpha per sample -> [N]
                alpha_t = alpha.gather(0, targets)

        loss = alpha_t * focal_term * ce_loss

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss

In [6]:
"""
DUKE 3D CNN MULTIMODAL BBOX MODEL
=================================
Goal:
- Best 3D IoU for tumor localization
- Single lesion per patient (uses first annotation bbox)

Inputs:
- 3D MRI volume (sampled from DICOM slices)
- Clinical features (including tumor grade as input feature)
- Radiomic imaging features

Output:
- 3D bounding box: [x_min, y_min, z_min, x_max, y_max, z_max] normalized in [0,1]
"""

import json
import warnings
from pathlib import Path

import numpy as np
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import matplotlib.patches as patches

warnings.filterwarnings("ignore")


# ==================== DATASET ====================

class Duke3DBBoxDataset(Dataset):
    """
    3D BBox dataset:
    - Builds a fixed-depth volume from DICOM slices
    - Uses first bbox annotation (single-lesion assumption)
    - Uses clinical + radiomic features
    - Adds tumor grade as a clinical input feature
    """

    def __init__(
        self,
        patients_dir,
        depth=32,
        image_size=128,
        use_clinical=True,
        use_imaging_features=True,
        use_grade_as_feature=True,
        augment=False
    ):
        self.patients_dir = Path(patients_dir)
        self.depth = depth
        self.image_size = image_size
        self.use_clinical = use_clinical
        self.use_imaging_features = use_imaging_features
        self.use_grade_as_feature = use_grade_as_feature
        self.augment = augment

        self.samples = []
        self.clinical_dim = 0
        self.imaging_dim = 0

        self._load_dataset()

    def _load_dataset(self):
        patient_folders = [d for d in self.patients_dir.iterdir() if d.is_dir()]
        clinical_features_list = []
        imaging_features_list = []

        valid_samples = 0
        with_bbox = 0

        for patient_dir in sorted(patient_folders):
            data_file = patient_dir / "patient_data.json"
            if not data_file.exists():
                continue

            try:
                with open(data_file) as f:
                    metadata = json.load(f)

                mri_dir = patient_dir / "MRI_DICOM_sample"
                if not mri_dir.exists():
                    continue

                dicom_files = sorted(list(mri_dir.glob("**/*.dcm")))
                if not dicom_files:
                    continue

                annotations = metadata.get("annotations", [])
                bbox_3d = None
                has_bbox = False

                if annotations and len(annotations) > 0:
                    ann = annotations[0]
                    if "bounding_box_3d" in ann and isinstance(ann["bounding_box_3d"], dict):
                        bbox_3d = ann["bounding_box_3d"]
                        x_min = bbox_3d.get("start_column", 0)
                        x_max = bbox_3d.get("end_column", 0)
                        y_min = bbox_3d.get("start_row", 0)
                        y_max = bbox_3d.get("end_row", 0)
                        if x_max > x_min and y_max > y_min:
                            has_bbox = True
                            with_bbox += 1

                if not has_bbox:
                    continue  # IoU-focused training: keep only valid bbox samples

                clinical = self._extract_clinical_features(metadata)
                imaging = self._extract_imaging_features(metadata)

                sample = {
                    "patient_id": patient_dir.name,
                    "dicom_files": dicom_files,
                    "clinical_features": clinical,
                    "imaging_features": imaging,
                    "bbox_3d": bbox_3d
                }
                self.samples.append(sample)
                valid_samples += 1

                if clinical is not None:
                    clinical_features_list.append(clinical)
                if imaging is not None:
                    imaging_features_list.append(imaging)

            except Exception:
                continue

        if clinical_features_list:
            self.clinical_mean = np.mean(clinical_features_list, axis=0)
            self.clinical_std = np.std(clinical_features_list, axis=0) + 1e-8
            self.clinical_dim = len(clinical_features_list[0])
        else:
            self.clinical_mean = np.zeros(1, dtype=np.float32)
            self.clinical_std = np.ones(1, dtype=np.float32)
            self.clinical_dim = 1

        if imaging_features_list:
            self.imaging_mean = np.mean(imaging_features_list, axis=0)
            self.imaging_std = np.std(imaging_features_list, axis=0) + 1e-8
            self.imaging_dim = len(imaging_features_list[0])
        else:
            self.imaging_mean = np.zeros(1, dtype=np.float32)
            self.imaging_std = np.ones(1, dtype=np.float32)
            self.imaging_dim = 1

        print("\nDataset Statistics:")
        print(f"  Total samples: {valid_samples}")
        print(f"  Samples with valid 3D bbox: {with_bbox}")

    def _extract_clinical_features(self, metadata):
        if not self.use_clinical:
            return None

        clinical = metadata.get("demographic_clinical", {})
        if not clinical:
            return None

        feature_keys = [
            "Age at last contact in EMR f/u(days)(from the date of diagnosis) ,last time patient known to be alive, unless age of death is reported(in such case the age of death",
            "Menopause (at diagnosis)",
            "Days to MRI (From the Date of Diagnosis)",
            "Field Strength (Tesla)",
            "TR (Repetition Time)",
            "TE (Echo Time)",
            "Slice Thickness ",
            "Staging(Nodes)#(Nx replaced by -1)[N]",
            "Staging(Metastasis)#(Mx -replaced by -1)[M]",
            "Staging(Tumor Size)#[T]"
        ]

        features = []
        for key in feature_keys:
            val = clinical.get(key, 0)
            if isinstance(val, (int, float)) and not np.isnan(val):
                features.append(float(val))
            else:
                features.append(0.0)

        if self.use_grade_as_feature:
            grade = clinical.get("Tumor Grade", 3)
            if not isinstance(grade, (int, float)) or grade not in [1, 2, 3]:
                grade = 3
            grade_norm = (float(grade) - 1.0) / 2.0  # 1->0.0, 2->0.5, 3->1.0
            features.append(grade_norm)

        return np.array(features, dtype=np.float32)

    def _extract_imaging_features(self, metadata):
        if not self.use_imaging_features:
            return None

        imaging = metadata.get("imaging_features", {})
        if not imaging:
            return None

        feature_keys = [
            "TumorMajorAxisLength_mm",
            "Volume_cu_mm_Tumor",
            "Energy_Tumor",
            "Contrast_Tumor",
            "Homogeneity1_Tumor",
            "Max_Enhancement_from_char_curv",
            "Time_to_Peak_from_char_curv",
            "Uptake_rate_from_char_curv",
            "Washout_rate_from_char_curv",
            "breastDensity_T1",
            "breastDensity_PostCon",
            "Peak_SER_tumor"
        ]

        features = []
        for key in feature_keys:
            val = imaging.get(key, 0)
            if isinstance(val, (int, float)) and not np.isnan(val):
                features.append(float(val))
            else:
                features.append(0.0)

        return np.array(features, dtype=np.float32)

    def _load_dicom_image(self, dicom_path):
        try:
            dcm = pydicom.dcmread(str(dicom_path))
            image = dcm.pixel_array

            while len(image.shape) > 2 and image.shape[0] == 1:
                image = image.squeeze(0)

            if len(image.shape) == 3:
                if image.shape[0] < image.shape[-1]:
                    image = image[image.shape[0] // 2]
                else:
                    if image.shape[-1] in [1, 3, 4]:
                        image = image[..., 0]
                    else:
                        image = image[:, :, image.shape[-1] // 2]

            if len(image.shape) != 2:
                return None

            image = image.astype(np.float32)
            if image.min() == image.max():
                return np.zeros_like(image, dtype=np.uint8)

            image = (image - image.min()) / (image.max() - image.min())
            image = (image * 255.0).astype(np.uint8)
            return image
        except Exception:
            return None

    def _build_volume(self, dicom_files):
        n = len(dicom_files)
        if n == 0:
            return torch.zeros((1, self.depth, self.image_size, self.image_size), dtype=torch.float32), 0

        indices = np.linspace(0, n - 1, self.depth, dtype=int)
        slices = []

        for idx in indices:
            img = self._load_dicom_image(dicom_files[idx])
            if img is None:
                img = np.zeros((self.image_size, self.image_size), dtype=np.uint8)
            else:
                img = np.array(
                    Image.fromarray(img, mode="L").resize((self.image_size, self.image_size), Image.BILINEAR),
                    dtype=np.uint8
                )
            slices.append(img)

        vol = np.stack(slices, axis=0).astype(np.float32) / 255.0  # [D,H,W]

        # Optional light augmentation
        if self.augment and np.random.rand() < 0.5:
            vol = vol[:, :, ::-1].copy()  # horizontal flip

        # Per-volume standardization
        mean, std = vol.mean(), vol.std() + 1e-6
        vol = (vol - mean) / std

        vol = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)  # [1,D,H,W]
        return vol, n

    def _create_bbox_target(self, bbox_3d, num_slices_original):
        """
        Target format: [x_min,y_min,z_min,x_max,y_max,z_max] normalized [0,1]
        
        CRITICAL FIX: num_slices_original is from the original volume (~100 slices), but 
        the model sees a resampled volume (self.depth=32 slices). We must remap 
        z-coordinates from original space to resampled space!
        """
        if bbox_3d is None or not isinstance(bbox_3d, dict):
            return torch.zeros(6, dtype=torch.float32)

        try:
            x_min = float(bbox_3d.get("start_column", 0))
            x_max = float(bbox_3d.get("end_column", 0))
            y_min = float(bbox_3d.get("start_row", 0))
            y_max = float(bbox_3d.get("end_row", 0))

            z_min_key = bbox_3d.get("start_slice") or bbox_3d.get("start_image", 0)
            z_max_key = bbox_3d.get("end_slice") or bbox_3d.get("end_image", num_slices_original - 1)
            z_min = float(z_min_key) if z_min_key is not None else 0.0
            z_max = float(z_max_key) if z_max_key is not None else float(num_slices_original - 1)

            # Validate basic bounds
            if x_max <= x_min or y_max <= y_min or z_max <= z_min:
                return torch.zeros(6, dtype=torch.float32)

            # Robust remap in sampled-index space (matches np.linspace used in _build_volume)
            sampled_indices = np.linspace(0, max(num_slices_original - 1, 0), self.depth)
            z_min_resampled = float(np.searchsorted(sampled_indices, z_min, side="left"))
            z_max_resampled = float(np.searchsorted(sampled_indices, z_max, side="right") - 1)

            # Clamp to valid range in resampled space
            z_min_resampled = float(np.clip(z_min_resampled, 0, self.depth - 1))
            z_max_resampled = float(np.clip(z_max_resampled, 0, self.depth - 1))

            # Keep valid thickness even if annotation is thinner than sampling interval
            if z_max_resampled <= z_min_resampled:
                if self.depth > 1:
                    z_max_resampled = min(z_min_resampled + 1.0, self.depth - 1)
                    z_min_resampled = max(0.0, z_max_resampled - 1.0)
                else:
                    return torch.zeros(6, dtype=torch.float32)

            # Normalize to [0, 1] using resampled depth
            bbox = torch.tensor([
                x_min / 512.0,
                y_min / 512.0,
                z_min_resampled / (self.depth - 1),
                x_max / 512.0,
                y_max / 512.0,
                z_max_resampled / (self.depth - 1)
            ], dtype=torch.float32)

            return torch.clamp(bbox, 0.0, 1.0)
        except Exception:
            return torch.zeros(6, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        volume, original_num_slices = self._build_volume(s["dicom_files"])
        bbox = self._create_bbox_target(s["bbox_3d"], original_num_slices)
        has_bbox = torch.tensor(1.0 if bbox.sum() > 0 else 0.0, dtype=torch.float32)

        clinical = s["clinical_features"]
        imaging = s["imaging_features"]

        if clinical is not None and self.use_clinical:
            clinical = (clinical - self.clinical_mean) / self.clinical_std
            clinical = torch.tensor(clinical, dtype=torch.float32)
        else:
            clinical = torch.zeros(self.clinical_dim, dtype=torch.float32)

        if imaging is not None and self.use_imaging_features:
            imaging = (imaging - self.imaging_mean) / self.imaging_std
            imaging = torch.tensor(imaging, dtype=torch.float32)
        else:
            imaging = torch.zeros(self.imaging_dim, dtype=torch.float32)

        return {
            "volume": volume,                  # [1,D,H,W]
            "clinical": clinical,              # [C]
            "imaging": imaging,                # [I]
            "bbox_3d": bbox,                   # [6]
            "has_bbox": has_bbox,              # scalar
            "patient_id": s["patient_id"]
        }


# ==================== MODEL ====================

class Conv3DBackbone(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),  # D/2 H/2 W/2

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),  # D/4 H/4 W/4

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),  # D/8 H/8 W/8

            nn.Conv3d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )

    def forward(self, x):
        x = self.features(x)  # [B,256,1,1,1]
        return x.flatten(1)   # [B,256]


class BBox3DMultimodalModel(nn.Module):
    """
    Predicts bbox via center-size parameterization for valid boxes:
    - center = sigmoid(raw_center) in [0,1]
    - size   = sigmoid(raw_size)   in [0,1]
    - bbox decoded as [c-s/2, c+s/2]
    """

    def __init__(self, clinical_dim, imaging_dim):
        super().__init__()

        self.image_encoder = Conv3DBackbone(in_channels=1)

        tab_dim = clinical_dim + imaging_dim
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tab_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True)
        )

        self.fusion = nn.Sequential(
            nn.Linear(256 + 128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True)
        )

        self.center_head = nn.Linear(128, 3)   # cx, cy, cz
        self.size_head = nn.Linear(128, 3)     # sx, sy, sz

    @staticmethod
    def decode_bbox(center_raw, size_raw):
        center = torch.sigmoid(center_raw)           # [B,3]
        size = torch.sigmoid(size_raw)               # [B,3], bounded
        half = size * 0.5

        mins = torch.clamp(center - half, 0.0, 1.0)
        maxs = torch.clamp(center + half, 0.0, 1.0)

        # enforce min <= max
        mins = torch.minimum(mins, maxs)
        maxs = torch.maximum(mins, maxs)

        bbox = torch.cat([mins[:, 0:1], mins[:, 1:2], mins[:, 2:3],
                          maxs[:, 0:1], maxs[:, 1:2], maxs[:, 2:3]], dim=1)
        return bbox

    def forward(self, volume, clinical, imaging):
        img_feat = self.image_encoder(volume)
        tab_feat = self.tabular_encoder(torch.cat([clinical, imaging], dim=1))
        fused = self.fusion(torch.cat([img_feat, tab_feat], dim=1))

        center_raw = self.center_head(fused)
        size_raw = self.size_head(fused)
        bbox = self.decode_bbox(center_raw, size_raw)

        return {
            "bbox_3d": bbox,
            "center_raw": center_raw,
            "size_raw": size_raw
        }


# ==================== TRAINER ====================

class BBox3DTrainer:
    def __init__(self, model, device, lr=1e-4, lambda_l1=0.4, lambda_iou=0.6):
        self.model = model
        self.device = device
        self.lambda_l1 = lambda_l1
        self.lambda_iou = lambda_iou

        self.l1 = nn.SmoothL1Loss(reduction="none")

        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="max", factor=0.5, patience=4
        )

        self.scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

        self.history = {
            "train_loss": [],
            "val_loss": [],
            "train_iou": [],
            "val_iou": [],
            "train_avg_pred": [],
            "val_avg_pred": []
        }
        self.best_iou = -1.0

    @staticmethod
    def bbox_iou_3d(pred, target):
        # pred/target: [B,6]
        px1, py1, pz1, px2, py2, pz2 = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3], pred[:, 4], pred[:, 5]
        tx1, ty1, tz1, tx2, ty2, tz2 = target[:, 0], target[:, 1], target[:, 2], target[:, 3], target[:, 4], target[:, 5]

        ix1 = torch.max(px1, tx1)
        iy1 = torch.max(py1, ty1)
        iz1 = torch.max(pz1, tz1)
        ix2 = torch.min(px2, tx2)
        iy2 = torch.min(py2, ty2)
        iz2 = torch.min(pz2, tz2)

        iw = torch.clamp(ix2 - ix1, min=0.0)
        ih = torch.clamp(iy2 - iy1, min=0.0)
        idd = torch.clamp(iz2 - iz1, min=0.0)
        inter = iw * ih * idd

        pvol = torch.clamp(px2 - px1, min=0.0) * torch.clamp(py2 - py1, min=0.0) * torch.clamp(pz2 - pz1, min=0.0)
        tvol = torch.clamp(tx2 - tx1, min=0.0) * torch.clamp(ty2 - ty1, min=0.0) * torch.clamp(tz2 - tz1, min=0.0)
        union = pvol + tvol - inter + 1e-6

        return inter / union  # [B]

    def _compute_loss(self, pred_bbox, gt_bbox, has_bbox):
        # mask only valid samples
        mask = has_bbox.float()  # [B]
        denom = torch.clamp(mask.sum(), min=1.0)

        l1_per = self.l1(pred_bbox, gt_bbox).mean(dim=1)  # [B]
        l1_loss = (l1_per * mask).sum() / denom

        iou = self.bbox_iou_3d(pred_bbox, gt_bbox)        # [B]
        iou_loss = ((1.0 - iou) * mask).sum() / denom

        total = self.lambda_l1 * l1_loss + self.lambda_iou * iou_loss
        return total, l1_loss.detach(), iou.detach()
    
    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0.0
        all_iou = []
        pred_sum = torch.zeros(6, device=self.device)
        pred_count = 0
    
        for batch in loader:
            volume = batch["volume"].to(self.device, non_blocking=True)
            clinical = batch["clinical"].to(self.device, non_blocking=True)
            imaging = batch["imaging"].to(self.device, non_blocking=True)
            bbox_gt = batch["bbox_3d"].to(self.device, non_blocking=True)
            has_bbox = batch["has_bbox"].to(self.device, non_blocking=True)
    
            self.optimizer.zero_grad(set_to_none=True)
    
            with torch.cuda.amp.autocast(enabled=(self.device.type == "cuda")):
                out = self.model(volume, clinical, imaging)
                pred_bbox = out["bbox_3d"]
                loss, _, iou = self._compute_loss(pred_bbox, bbox_gt, has_bbox)
    
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
    
            total_loss += loss.item()
    
            valid = has_bbox > 0
            if valid.any():
                all_iou.extend(iou[valid].detach().cpu().tolist())
                pred_sum += pred_bbox[valid].detach().sum(dim=0)
                pred_count += int(valid.sum().item())
    
        train_loss = total_loss / len(loader)
        train_iou = float(np.mean(all_iou)) if len(all_iou) else 0.0
        avg_pred = (pred_sum / max(pred_count, 1)).detach().cpu().numpy().tolist()
    
        return train_loss, train_iou, avg_pred

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        total_loss = 0.0
        all_iou = []
        pred_sum = torch.zeros(6, device=self.device)
        pred_count = 0
    
        for batch in loader:
            volume = batch["volume"].to(self.device, non_blocking=True)
            clinical = batch["clinical"].to(self.device, non_blocking=True)
            imaging = batch["imaging"].to(self.device, non_blocking=True)
            bbox_gt = batch["bbox_3d"].to(self.device, non_blocking=True)
            has_bbox = batch["has_bbox"].to(self.device, non_blocking=True)
    
            out = self.model(volume, clinical, imaging)
            pred_bbox = out["bbox_3d"]
            loss, _, iou = self._compute_loss(pred_bbox, bbox_gt, has_bbox)
    
            total_loss += loss.item()
    
            valid = has_bbox > 0
            if valid.any():
                all_iou.extend(iou[valid].cpu().tolist())
                pred_sum += pred_bbox[valid].sum(dim=0)
                pred_count += int(valid.sum().item())
    
        val_loss = total_loss / len(loader)
        val_iou = float(np.mean(all_iou)) if len(all_iou) else 0.0
        avg_pred = (pred_sum / max(pred_count, 1)).detach().cpu().numpy().tolist()
    
        return val_loss, val_iou, avg_pred

    def train(self, train_loader, val_loader, num_epochs=50, patience=10):
        print("\n" + "=" * 70)
        print("3D BBOX TRAINING")
        print("=" * 70)

        no_improve = 0
        for epoch in range(num_epochs):
            train_loss, train_iou, train_avg_pred = self.train_epoch(train_loader)
            val_loss, val_iou, val_avg_pred = self.validate(val_loader)
        
            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_iou"].append(train_iou)
            self.history["val_iou"].append(val_iou)
            self.history["train_avg_pred"].append(train_avg_pred)
            self.history["val_avg_pred"].append(val_avg_pred)

            self.scheduler.step(val_iou)
        
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print(f"  Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f}")
            print(f"  Val Loss:   {val_loss:.4f} | Val IoU:   {val_iou:.4f}")
            print(f"  Avg Pred Train [xmin,ymin,zmin,xmax,ymax,zmax]: {[round(x, 3) for x in train_avg_pred]}")
            print(f"  Avg Pred Val   [xmin,ymin,zmin,xmax,ymax,zmax]: {[round(x, 3) for x in val_avg_pred]}")
        
            if val_iou > self.best_iou:
                self.best_iou = val_iou
                no_improve = 0
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": self.model.state_dict(),
                        "best_iou": self.best_iou
                    },
                    "best_3d_bbox_model.pth"
                )
                print("  Best model saved!")
            else:
                no_improve += 1
                print(f"  Patience: {no_improve}/{patience}")
        
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        ckpt = torch.load("best_3d_bbox_model.pth", map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        print(f"\nTraining complete! Best Val IoU: {self.best_iou:.4f}")


# ==================== VISUALIZATION ====================

@torch.no_grad()
def visualize_predictions(model, loader, device, num_samples=6):
    model.eval()
    shown = 0
    fig = plt.figure(figsize=(14, 4 * num_samples))

    for batch in loader:
        volume = batch["volume"].to(device)
        clinical = batch["clinical"].to(device)
        imaging = batch["imaging"].to(device)
        gt = batch["bbox_3d"]
        ids = batch["patient_id"]

        out = model(volume, clinical, imaging)
        pred = out["bbox_3d"].cpu()

        # show middle slice (for quick XY view)
        vols = batch["volume"]  # [B,1,D,H,W]
        mid_idx = vols.shape[2] // 2

        for i in range(vols.shape[0]):
            if shown >= num_samples:
                break

            img = vols[i, 0, mid_idx].numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            ax1 = plt.subplot(num_samples, 2, shown * 2 + 1)
            ax1.imshow(img, cmap="gray")
            ax1.set_title(f"Patient: {ids[i]} (mid-slice)")
            ax1.axis("off")

            ax2 = plt.subplot(num_samples, 2, shown * 2 + 2)
            ax2.imshow(img, cmap="gray")

            # draw XY box on rendered view
            gt_box = gt[i].numpy()
            pr_box = pred[i].numpy()

            gx1, gy1, _, gx2, gy2, _ = gt_box
            px1, py1, _, px2, py2, _ = pr_box

            h, w = img.shape
            gx1 *= w; gy1 *= h; gx2 *= w; gy2 *= h
            px1 *= w; py1 *= h; px2 *= w; py2 *= h

            gx1 = np.clip(gx1, 0, w - 1); gx2 = np.clip(gx2, 0, w - 1)
            gy1 = np.clip(gy1, 0, h - 1); gy2 = np.clip(gy2, 0, h - 1)
            px1 = np.clip(px1, 0, w - 1); px2 = np.clip(px2, 0, w - 1)
            py1 = np.clip(py1, 0, h - 1); py2 = np.clip(py2, 0, h - 1)

            gt_valid = bool(gx2 > gx1 and gy2 > gy1)
            pred_valid = bool(px2 > px1 and py2 > py1)

            if gt_valid:
                gw, gh = gx2 - gx1, gy2 - gy1
                ax2.add_patch(patches.Rectangle((gx1, gy1), gw, gh, fill=False, edgecolor="green", linewidth=2, label="GT"))
            else:
                ax2.text(0.02, 0.95, "GT projected box is empty", color="yellow", fontsize=8,
                         transform=ax2.transAxes, va="top", ha="left",
                         bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5))

            if pred_valid:
                pw, ph = px2 - px1, py2 - py1
                ax2.add_patch(patches.Rectangle((px1, py1), pw, ph, fill=False, edgecolor="red", linewidth=2, linestyle="--", label="Pred"))

            z_text = f"GT z:[{gt_box[2]:.2f},{gt_box[5]:.2f}]  Pred z:[{pr_box[2]:.2f},{pr_box[5]:.2f}]"
            ax2.set_title("XY BBox (Green=GT, Red=Pred)\n" + z_text)
            ax2.axis("off")
            handles, _ = ax2.get_legend_handles_labels()
            if handles:
                ax2.legend(loc="upper right", fontsize=8)

            shown += 1

        if shown >= num_samples:
            break

    plt.tight_layout()
    plt.savefig("3d_bbox_predictions.png", dpi=250, bbox_inches="tight")
    plt.close()
    print("Saved: 3d_bbox_predictions.png")


def plot_training(history):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(history["train_loss"], label="Train Loss")
    ax[0].plot(history["val_loss"], label="Val Loss")
    ax[0].set_title("Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].legend()
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(history["train_iou"], label="Train IoU", color="blue")
    ax[1].plot(history["val_iou"], label="Val IoU", color="green")
    ax[1].set_title("3D IoU")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("3d_bbox_training_curves.png", dpi=250, bbox_inches="tight")
    plt.close()
    print("Saved: 3d_bbox_training_curves.png")


# ==================== MAIN ====================

def main():
    print("\n" + "=" * 70)
    print("DUKE 3D CNN BBOX DETECTION (MULTIMODAL)")
    print("=" * 70)

    export_dir = "exported_patients"
    batch_size = 2   # 3D model; increase if memory allows
    num_epochs = 60
    lr = 1e-4
    depth = 32
    image_size = 128
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Device: {device}")
    print(f"Batch Size: {batch_size}")
    print(f"Depth x Size: {depth} x {image_size} x {image_size}")

    print("\n" + "=" * 70)
    print("LOADING DATASETS")
    print("=" * 70)

    train_ds = Duke3DBBoxDataset(
        Path(export_dir) / "train",
        depth=depth,
        image_size=image_size,
        use_clinical=True,
        use_imaging_features=True,
        use_grade_as_feature=True,
        augment=True
    )

    val_ds = Duke3DBBoxDataset(
        Path(export_dir) / "test",
        depth=depth,
        image_size=image_size,
        use_clinical=True,
        use_imaging_features=True,
        use_grade_as_feature=True,
        augment=False
    )

    # share normalization stats
    val_ds.clinical_mean = train_ds.clinical_mean
    val_ds.clinical_std = train_ds.clinical_std
    val_ds.imaging_mean = train_ds.imaging_mean
    val_ds.imaging_std = train_ds.imaging_std
    val_ds.clinical_dim = train_ds.clinical_dim
    val_ds.imaging_dim = train_ds.imaging_dim

    print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)}")
    print(f"Clinical dim: {train_ds.clinical_dim}")
    print(f"Imaging dim: {train_ds.imaging_dim}")

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True
    )

    print("\n" + "=" * 70)
    print("INITIALIZING MODEL")
    print("=" * 70)

    model = BBox3DMultimodalModel(
        clinical_dim=train_ds.clinical_dim,
        imaging_dim=train_ds.imaging_dim
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    trainer = BBox3DTrainer(
        model=model,
        device=device,
        lr=lr,
        lambda_l1=0.4,
        lambda_iou=0.6
    )

    trainer.train(train_loader, val_loader, num_epochs=num_epochs, patience=12)

    print("\n" + "=" * 70)
    print("EVALUATION + VISUALIZATION")
    print("=" * 70)

    plot_training(trainer.history)
    visualize_predictions(model, val_loader, device, num_samples=6)

    print("\nFinished!")
    print("Best model: best_3d_bbox_model.pth")
    print("Curves: 3d_bbox_training_curves.png")
    print("Predictions: 3d_bbox_predictions.png")
    print(f"Best Val IoU: {trainer.best_iou:.4f}")


if __name__ == "__main__":
    main()


DUKE 3D CNN BBOX DETECTION (MULTIMODAL)
Device: cuda
Batch Size: 2
Depth x Size: 32 x 128 x 128

LOADING DATASETS

Dataset Statistics:
  Total samples: 671
  Samples with valid 3D bbox: 671

Dataset Statistics:
  Total samples: 169
  Samples with valid 3D bbox: 169

Train: 671 | Val: 169
Clinical dim: 11
Imaging dim: 12

INITIALIZING MODEL
Total parameters: 1,315,334

3D BBOX TRAINING

Epoch 1/60
  Train Loss: 0.6091 | Train IoU: 0.0008
  Val Loss:   0.6026 | Val IoU:   0.0055
  Avg Pred Train [xmin,ymin,zmin,xmax,ymax,zmax]: [0.38, 0.422, 0.779, 0.552, 0.667, 0.965]
  Avg Pred Val   [xmin,ymin,zmin,xmax,ymax,zmax]: [0.324, 0.406, 0.947, 0.424, 0.553, 1.0]
  Best model saved!

Epoch 2/60
  Train Loss: 0.5928 | Train IoU: 0.0265
  Val Loss:   0.5843 | Val IoU:   0.0414
  Avg Pred Train [xmin,ymin,zmin,xmax,ymax,zmax]: [0.23, 0.494, 0.964, 0.348, 0.685, 1.0]
  Avg Pred Val   [xmin,ymin,zmin,xmax,ymax,zmax]: [0.184, 0.478, 0.971, 0.314, 0.687, 1.0]
  Best model saved!

Epoch 3/60
  Train